# Ticket 2: Feature Engineering Time (Đặc trưng Thời gian & Lag / Rolling)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN
Mục tiêu của Notebook này là trích xuất các đặc trưng theo thời gian (Time-based Features) để mô hình Machine Learning có thể bắt được các quy luật chu kỳ, mùa vụ và xu hướng gần nhất. Chúng ta cần tuân thủ nghiêm ngặt:
- **Nguyên tắc Point-In-Time**: Chống rò rỉ dữ liệu (Lookahead Bias / Data Leakage) ở mức cao nhất, mô hình tại thời điểm t không được biết dữ liệu ở t+1.
- **Vectorization**: Tránh `for loop` hoặc `apply`, sử dụng tích hợp sẵn Pandas/PyArrow để tránh bottleneck.
- **Historical Context**: Giữ lại dữ liệu liền trước của Train để làm context cho tập Validation (cho tính Lag/Rolling) tránh NaN.

Cấu trúc Notebook được chia làm 5 bước chính theo chuẩn của dự án.

## Bước 1. Import Libraries & Load Data
Nạp dữ liệu thô và thư viện cần thiết. Kiểm tra các cột bắt buộc: `timestamp`, `energy_generated_kwh`, `site_id`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Import custom feature engineering module
from fe_temporal import build_time_features, build_lag_rolling_features

import lightgbm as lgb
from sklearn.metrics import mean_squared_error

# Cấu hình visualization
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)


In [ ]:
# Lưu ý: Bạn cần thay đổi đường dẫn này trỏ tới tập dữ liệu thực tế của bạn
# Ví dụ tạo dummy data để minh hoạ:
dates = pd.date_range(start='2024-01-01', end='2024-02-01', freq='15T')
site_ids = [1, 2]

dummy_data = []
for site in site_ids:
    df_site = pd.DataFrame({'timestamp': dates})
    df_site['site_id'] = site
    # Simulate some energy pattern
    df_site['energy_generated_kwh'] = np.where(
        (df_site.timestamp.dt.hour >= 6) & (df_site.timestamp.dt.hour <= 18),
        np.random.normal(50, 10, size=len(df_site)), 
        0
    )
    dummy_data.append(df_site)

df = pd.concat(dummy_data, ignore_index=True)
df = df.sort_values(['site_id', 'timestamp']).reset_index(drop=True)

print("Sample Data:")
display(df.head())
print("Cột có sẵn:", df.columns.tolist())

## Bước 2. Time-Based Train/Val/Test Split
Chia tập dữ liệu theo mốc thời gian. Giữ lại một khoảng lịch sử ngắn (vd: 96 bước thời gian cuối của Train) làm `context` cho tập Validation để khi tính toán Lag & Rolling cho các dòng đầu của Validation không bị lỗi `NaN`.

In [ ]:
# Sắp xếp lại dữ liệu toàn cục theo thời gian (nếu cần)
df = df.sort_values(['timestamp', 'site_id']).reset_index(drop=True)

# Giả sử chia train/val/test theo ngày
train_end = '2024-01-20'
val_end = '2024-01-25'

df_train = df[df['timestamp'] < train_end].copy()
df_val_raw = df[(df['timestamp'] >= train_end) & (df['timestamp'] < val_end)].copy()
df_test_raw = df[df['timestamp'] >= val_end].copy()

# Lấy context từ train (ví dụ 96 steps = 24h)
context_steps = 96
df_train_context = df_train.groupby('site_id').tail(context_steps)
df_val = pd.concat([df_train_context, df_val_raw]).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

# Lấy context từ val cho test
df_val_context = df_val_raw.groupby('site_id').tail(context_steps)
df_test = pd.concat([df_val_context, df_test_raw]).sort_values(['site_id', 'timestamp']).reset_index(drop=True)

print(f"Train size: {len(df_train)}, Val size (with context): {len(df_val)}, Test size (with context): {len(df_test)}")

## Bước 3. Feature Generation
Áp dụng các hàm tạo đặc trưng thời gian và đặc trưng trễ (Lag/Rolling) từ module `fe_temporal.py`. Đảm bảo các hàm này tuân thủ nguyên tắc Point-in-time và tối ưu Vectorization.

In [ ]:
def apply_feature_engineering(data):
    # 1. Tạo đặc trưng thời gian chu kỳ & lịch
    data = build_time_features(data, timestamp_col='timestamp')
    # 2. Tạo đặc trưng trễ và trung bình trượt
    data = build_lag_rolling_features(data, group_col='site_id', target_col='energy_generated_kwh')
    return data

df_train_fe = apply_feature_engineering(df_train)
df_val_fe = apply_feature_engineering(df_val)
df_test_fe = apply_feature_engineering(df_test)

# Bỏ đi các dòng context trong val/test để đánh giá chính xác
# Vì ta đã tính xong Lag/Rolling, ta chỉ giữ lại những dòng thuộc phân khúc val/test thực sự
df_val_fe = df_val_fe[df_val_fe['timestamp'] >= train_end].reset_index(drop=True)
df_test_fe = df_test_fe[df_test_fe['timestamp'] >= val_end].reset_index(drop=True)

df_train_fe = df_train_fe.dropna().reset_index(drop=True) # Drop NaN sinh ra ở các dòng đầu của train do lag

print("Các feature được tạo ra:")
print(df_train_fe.columns.tolist())

## Bước 4. Verification Pipeline (Kiểm chứng Phần B)
Quy trình kiểm chứng bắt buộc để phát hiện hiện tượng Persistence Model / Echo Effect (khiến đường dự báo y_pred bị dịch sang phải 1 nhịp so với y_true).

**4.1. Huấn luyện mô hình LightGBM (Baseline vs Full)**
Mô hình dùng nhóm features có chứa Lag/Rolling (Full) so với không có (Baseline).

In [ ]:
features = [c for c in df_train_fe.columns if c not in ['timestamp', 'energy_generated_kwh', 'site_id', 'season']]
target = 'energy_generated_kwh'

X_train, y_train = df_train_fe[features], df_train_fe[target]
X_val, y_val = df_val_fe[features], df_val_fe[target]

model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_val)
rmse = mean_squared_error(y_val, y_pred, squared=False)
print(f"Validation RMSE: {rmse:.4f}")

**4.2. Vẽ biểu đồ Bar Chart cho feature_importances_ và Đo % đóng góp**
Tính tổng phần trăm đóng góp của nhóm Lag/Rolling so với tổng các đặc trưng khác.

In [ ]:
importances = model.feature_importances_
feat_imp = pd.DataFrame({'feature': features, 'importance': importances})
feat_imp = feat_imp.sort_values(by='importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp, x='importance', y='feature', hue='feature', legend=False)
plt.title('Feature Importances')
plt.show()

# Tính tỷ lệ đóng góp của Lag/Rolling
lag_rolling_feats = [f for f in features if 'lag' in f or 'rolling' in f]
lag_rolling_imp = feat_imp[feat_imp['feature'].isin(lag_rolling_feats)]['importance'].sum()
total_imp = feat_imp['importance'].sum()
print(f"Tỷ lệ quan trọng của nhóm Lag/Rolling: {lag_rolling_imp / total_imp * 100:.2f}%")

**4.3. Vẽ Đồ thị Dự báo vs Thực tế (Overlay Plot)**
Zoom vào một khoảng thời gian 3 - 7 ngày trên tập Validation để xem đường y_pred có đi theo sau y_true không.

In [ ]:
df_plot = df_val_fe.copy()
df_plot['prediction'] = y_pred

# Plot 1 site và zoom khoảng 3 ngày đầu
site_plot = df_plot[df_plot['site_id'] == site_ids[0]].head(96 * 3)

plt.figure(figsize=(15, 5))
plt.plot(site_plot['timestamp'], site_plot['energy_generated_kwh'], label='True Value', marker='.')
plt.plot(site_plot['timestamp'], site_plot['prediction'], label='Prediction', alpha=0.8, marker='+')
plt.title(f"Overlay Plot (Site {site_ids[0]}) - Trực quan hóa để soi Echo Effect / Trễ nhịp")
plt.xlabel('Timestamp')
plt.ylabel('Energy (kWh)')
plt.legend()
plt.show()

**4.4 Quyết định lọc Feature**
Dựa trên đồ thị Overlay Plot và tỷ lệ quan trọng, ta thấy: Nếu Lag/Rolling > 50-60% mà gây trễ rệt thì phải loại bỏ. Ở đây nếu đồ thị bám sát và không bị Echo Effect, ta sẽ duyệt giữ lại các tính năng này.

## Bước 5. Export Processed Datasets
Xuất tập dữ liệu đã hoàn thiện ra Parquet hoặc CSV sẵn sàng cho bước Training tiếp theo.

In [ ]:
import os
output_dir = 'processed_data'
os.makedirs(output_dir, exist_ok=True)

# df_train_fe.to_parquet(f'{output_dir}/train_fe.parquet')
# df_val_fe.to_parquet(f'{output_dir}/val_fe.parquet')
# df_test_fe.to_parquet(f'{output_dir}/test_fe.parquet')

print("Đã lưu các datasets (ví dụ minh hoạ).")